# Lab 2.4 — HuggingFace in Depth
**Module II · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-2-llm/lab2_4_huggingface_deep_dive.ipynb)

---

## What you will do
Labs 2.1–2.3 used `SimpleLLM` — a wrapper that hides whether you are talking to Ollama or HuggingFace. This lab lifts the hood and shows you the HuggingFace Transformers library directly:

1. **`pipeline()`** — the one-liner entry point: run any HuggingFace model in three lines.
2. **Tokenization** — how text becomes token IDs, what special tokens look like, and how chat templates work.
3. **`AutoModelForCausalLM`** — the manual generation loop with full control over decoding strategy.
4. **`SentenceTransformer`** — sentence embeddings and semantic similarity, the backbone of the RAG system you built in Lab 2.3.

## Prerequisites
Lab 2.1 completed. Labs 2.2 and 2.3 recommended for context.

**Estimated time:** 70–90 min

---
## 0 · Setup

The model weights (~3.4 GB) are downloaded and cached the first time. If you already ran Lab 2.1 locally with the HuggingFace fallback, they are already on disk.

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                    "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

plt.rcParams["figure.dpi"] = 110

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
EMB_ID   = "sentence-transformers/all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports OK  |  PyTorch {torch.__version__}  |  device: {device}")

---
## 1 · The `pipeline()` API — the fast path

`pipeline()` is HuggingFace's one-liner for inference. You give it a task name and a model ID; it handles downloading, tokenization, inference, and decoding automatically.

```python
from transformers import pipeline
gen = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-1.7B-Instruct")
gen("Explain ML in one sentence:", max_new_tokens=60)
```

This is what `SimpleLLM` calls under the hood when Ollama is not available.

### Exercise 2.4.1 `[Core]` — Load a pipeline and chat with it

1. Load a `"text-generation"` pipeline for `MODEL_ID` with `device_map="auto"` and `torch_dtype=torch.bfloat16`.
2. Run a plain text-completion prompt (`return_full_text=False` to get only new tokens).
3. Run a chat-style prompt using the HuggingFace **messages format** — note how similar it looks to `llm.chat()` from Lab 2.1.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Load a "text-generation" pipeline for MODEL_ID with device_map="auto" and torch_dtype=torch.bfloat16.
raise NotImplementedError("Complete this exercise")

In [ ]:
# Plain text completion
prompt = "The key difference between supervised and unsupervised learning is"
result = gen_pipe(prompt, max_new_tokens=60, do_sample=False, return_full_text=False)
print(result[0]["generated_text"])

In [ ]:
# Chat messages — same dict format as llm.chat() in Lab 2.1
messages = [
    {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",   "content": "What is a transformer architecture?"},
]
result = gen_pipe(messages, max_new_tokens=80, do_sample=False)
# pipeline returns the full conversation; the last message is the model reply
print(result[0]["generated_text"][-1]["content"])

> **What you just did in ~5 lines** is what `SimpleLLM` wraps into `.generate()` and `.chat()`. The `pipeline()` is the right tool when defaults are acceptable. Section 3 shows the manual approach, which gives you access to raw logits, custom sampling, and integration with training loops.

---
## 2 · Tokenization — how text becomes numbers

LLMs operate on **tokens**, not characters or words. A tokenizer maps text to a sequence of integer IDs from a fixed vocabulary, and maps outputs back to text. Understanding this matters because:

- It explains why `llm.count_tokens()` from Lab 2.1 behaves non-uniformly across languages and content types.
- It determines what fits in the context window.
- Chat models wrap each message in **special role tokens** — `apply_chat_template()` does this automatically.

### Exercise 2.4.2 `[Core]` — Encode a sentence and inspect every token

Load `AutoTokenizer` and encode a sentence. Print the token string, its integer ID, and the attention mask value for each position. Then decode back to verify round-trip correctness.

In [ ]:
# YOUR CODE HERE
# Hint: Load AutoTokenizer and encode a sentence. Print the token string, its integer ID, and the attention mask value for ea...
raise NotImplementedError("Complete this exercise")

### Exercise 2.4.3 `[Core]` — Token counts vary by content

Tokenize several different strings and compare token counts. Then use `apply_chat_template()` to see what special role tokens the model actually receives when you call `.chat()`.

In [ ]:
# YOUR CODE HERE
# Hint: Tokenize several different strings and compare token counts. Then use apply_chat_template() to see what special role ...
raise NotImplementedError("Complete this exercise")

In [ ]:
# Chat template — what the model actually sees for a multi-turn conversation
messages = [
    {"role": "system",    "content": "You are a helpful assistant."},
    {"role": "user",      "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
    {"role": "user",      "content": "And Germany?"},
]
formatted = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
print(formatted)
print(f"\nTotal tokens: {len(tokenizer.encode(formatted))}")

> **What to notice:**
> - Tokenisation is sub-word: "Supercalifragilistic" splits into several pieces; "Hello!" may be one token.
> - The same sentence uses more tokens in Spanish — the tokenizer was optimised on English-heavy training data.
> - `apply_chat_template` wraps every message with role markers (`<|im_start|>system`, `<|im_end|>`, …). This is exactly what `SimpleLLM.chat()` does before sending to the model — now you can see the raw string.

---
## 3 · `AutoModelForCausalLM` — the manual generation loop

`pipeline()` abstracts tokenisation, model forward pass, and decoding into one call. Here we unpack each step. This level of control is useful when you need to:
- Inspect logits or token probabilities.
- Apply custom sampling logic.
- Integrate generation into a training or fine-tuning loop.

### Exercise 2.4.4 `[Core]` — Load the model and run greedy decoding

Load `AutoModelForCausalLM`, apply the chat template manually, tokenize, call `model.generate()` with greedy decoding (`do_sample=False`), and decode only the **new** tokens (not the prompt).

In [ ]:
# YOUR CODE HERE
# Hint: Load AutoModelForCausalLM, apply the chat template manually, tokenize, call model.generate() with greedy decoding (do...
raise NotImplementedError("Complete this exercise")

In [ ]:
messages = [
    {"role": "system", "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",   "content": "What is backpropagation?"},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=80, do_sample=False)

# Slice off the input tokens — keep only what the model generated
new_ids = output_ids[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_ids, skip_special_tokens=True))

### Exercise 2.4.5 `[Extension]` — Compare four generation strategies

Run the same open-ended prompt through four different decoding strategies and compare the outputs:

| Strategy | `do_sample` | Key parameter | Behaviour |
|---|---|---|---|
| Greedy | `False` | — | Always picks the highest-probability token |
| Top-k | `True` | `top_k=50` | Samples from the 50 most likely tokens |
| Top-p (nucleus) | `True` | `top_p=0.9` | Samples from the smallest set covering 90 % of probability mass |
| Beam search | `False` | `num_beams=4` | Keeps 4 candidate sequences and picks the globally best one |

In [ ]:
# YOUR CODE HERE
# Hint: Run the same open-ended prompt through four different decoding strategies and compare the outputs:
raise NotImplementedError("Complete this exercise")

> **What to notice:**
> - **Greedy** is deterministic but can fall into repetitive loops on open-ended prompts.
> - **Top-k / Top-p** add diversity while filtering very unlikely tokens; they are the default for creative tasks and the strategy used by `SimpleLLM` at `temperature > 0`.
> - **Beam search** produces more globally coherent text but uses 4× memory and can sound formulaic.
> - For factual tasks (RAG answers, Q&A), `do_sample=False` is preferred. For creative or conversational generation, `top_p=0.9, temperature=0.7` is a common starting point.

---
## 4 · Sentence embeddings — the backbone of RAG

In Lab 2.3 you used FAISS to retrieve relevant document chunks. Under the hood, every chunk was converted to a **dense vector** by a `SentenceTransformer`. Two sentences that mean the same thing — even using completely different words — end up close in embedding space. That is what makes semantic search work beyond simple keyword matching.

### Exercise 2.4.6 `[Core]` — Semantic similarity and retrieval

1. Encode six sentences with `SentenceTransformer`.
2. Compute the pairwise cosine similarity matrix and visualise it as a heatmap.
3. Run a query sentence and rank the six by similarity — this is exactly what the RAG retriever does.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Encode six sentences with SentenceTransformer.
raise NotImplementedError("Complete this exercise")

In [ ]:
sentences = [
    "S1: What is the return policy for electronics?",
    "S2: Electronics can be returned within 15 days.",
    "S3: Express shipping costs 39,900 COP.",
    "S4: How much does express delivery cost?",
    "S5: The weather in Bogotá is warm and sunny.",
    "S6: Customers may return smartphones within two weeks.",
]

# normalize_embeddings=True → dot product equals cosine similarity
embeddings = emb_model.encode(sentences, normalize_embeddings=True)
print(f"Embeddings shape: {embeddings.shape}")

sim = embeddings @ embeddings.T

fig, ax = plt.subplots(figsize=(8, 6))
labels = [s[:32] + "…" for s in sentences]
sns.heatmap(sim, annot=True, fmt=".2f", cmap="RdYlGn",
            xticklabels=[f"S{i+1}" for i in range(len(sentences))],
            yticklabels=labels, ax=ax, vmin=0, vmax=1)
ax.set_title("Pairwise cosine similarity — sentence embeddings")
plt.tight_layout()
plt.show()

In [ ]:
# Semantic retrieval — same operation as the FAISS index in Lab 2.3
query = "Can I return a phone I bought last week?"
q_emb = emb_model.encode([query], normalize_embeddings=True)
scores = (q_emb @ embeddings.T)[0]

print(f"Query: {query!r}\n")
print(f"{'Score':>6}  Sentence")
print("─" * 65)
for score, sent in sorted(zip(scores, sentences), reverse=True):
    print(f"{score:.3f}  {sent}")

> **What to notice:**
> - **S2 and S6** are near-paraphrases ("15 days" vs "two weeks") — the embedding model correctly places them very close despite the different wording.
> - **S3 and S4** cluster together (both about shipping cost) even though they share almost no words.
> - **S5** (weather) is far from everything else — correctly flagged as unrelated.
> - The query ("Can I return a phone...") ranks **S2 and S6 highest** — the correct policy sentences — without any keyword overlap. This semantic matching is the core mechanism of the RAG retriever in Lab 2.3.

---
## Summary

| What we covered | Key takeaway |
|---|---|
| `pipeline()` | One-liner inference — right for prototyping and simple deployments |
| `AutoTokenizer` | Text → sub-word token IDs; chat templates add role markers automatically |
| `AutoModelForCausalLM` | Full control: explicit tokenisation, configurable `model.generate()` |
| Generation strategies | Greedy = deterministic; top-k/p = diverse; beam = coherent. Pick by task |
| `SentenceTransformer` | Dense embeddings capture meaning, not keywords — the engine behind RAG |

---

## Connection to the rest of the course

You now see what `SimpleLLM` was abstracting in Labs 2.1–2.3. In **Module III** you will train Graph Neural Networks using the same `model.eval()` / forward-pass / loss pattern you saw in Section 3. In **Module IV** the sentence embedding model you used here is replaced by a graph-aware encoder — enabling retrieval over structured relational data, not just flat text.